# Registering conditions onto the features DataFrame

This extends the features table produced by `example_workflow.ipynb`
(one row per well, one column per feature) with per-well experiment
metadata from a layout file — cell line, treatment, dose, batch,
differentiation date, etc.

Well labels (A1..D6) repeat on every plate, so the join key has to be
**(recording, well)**, not `well` alone — merging on `well` by itself
would cross-wire conditions between recordings as soon as more than one
file is involved.

Real layout files (e.g. `layout/Bioni37A_3_050325.csv`) are one row per
well with explicit metadata columns, not the row/column-letter grid that
`MEA_features.read_layout` expects — so here the layout is just read
with pandas and merged onto `df`.

In [ ]:
import os

import pandas as pd

## 1) Load the features table from `example_workflow.ipynb`

Reuses that notebook's own output instead of recomputing features from
the raw recording.

In [ ]:
df = pd.read_csv(os.path.join('results', 'features_table.csv'))
df.shape

## 2) Load the layout

`recording_name` + `well` is the join key; `exclude` flags wells that
shouldn't be analyzed; `treatment`/`dose`/`dose_unit` describe the
condition.

In [ ]:
layout_path = os.path.join('layout', 'Bioni37A_3_050325.csv')
layout = pd.read_csv(layout_path)
layout[['recording_name', 'well', 'cell_line', 'treatment', 'dose', 'dose_unit', 'batch', 'div', 'exclude']].head()

## 3) Match each well to its recording

`df`'s `file_path` identifies which recording a row came from; derive
the same `recording_name` the layout uses from it, so each well can be
matched to *its own* recording's layout row rather than just any row
with the same well label.

In [ ]:
df['recording_name'] = df['file_path'].apply(lambda p: os.path.basename(p).rsplit('_spikes.csv', 1)[0])
df[['file_path', 'recording_name', 'well']].head()

## 4) Merge conditions onto df and drop excluded wells

In [ ]:
def make_condition(row):
    """Human-readable condition label, e.g. 'baseline' or 'LTX5 1uM'."""
    if pd.isna(row['dose']):
        return row['treatment']
    return f"{row['treatment']} {row['dose']:g}{row['dose_unit']}"

layout = layout.copy()
layout['condition'] = layout.apply(make_condition, axis=1)

condition_cols = ['recording_name', 'well', 'cell_line', 'condition', 'treatment', 'dose', 'dose_unit',
                  'batch', 'div', 'differentiation_date', 'recording_date', 'exclude']
df = df.merge(layout[condition_cols], on=['recording_name', 'well'], how='left')

# Every well should have matched a layout row - a left join leaves NaNs
# behind for any (recording, well) pair the layout doesn't cover.
unmatched = df['condition'].isna()
assert not unmatched.any(), (
    f"{unmatched.sum()} well(s) had no matching layout row:\n"
    f"{df.loc[unmatched, ['recording_name', 'well']]}"
)

# Layout-flagged exclusions shouldn't be analyzed
df = df[df['exclude'] != 'Yes'].drop(columns='exclude').reset_index(drop=True)

df[['recording_name', 'well', 'condition', 'cell_line', 'batch', 'div', 'ISI [s]', 'FR [Hz]']].head()

For many recording files at once, build `features_table.csv` across
all of them first (one row per well per recording, each with its own
`file_path`/`recording_name`), then do this same `(recording_name, well)`
merge once against the concatenated layout rows for every file. The
per-file processors in `MEA_features.py` (`process_recording_file`,
`process_multidrug_recording_file`, `process_plate_layout_recording_file`,
...) do this same per-file layout merge internally — see their
docstrings for which layout convention each one expects.